# 04 - Evaluate LLaMEA Champions (N=10 Independent Benchmark Runs)

This notebook executes the official empirical benchmark protocol for the best synthesized algorithms:
1. Loads winning Clean and Noisy champions from `data/champions.json` (discovered across all models from `db.sqlite3`).
2. **Pre-flight Audit Dashboard**: Scans `results/evaluations/` to display exact completion status, cached runs, pending runs, and required reruns.
3. Executes each pending champion across **$N=10$ independent problem instances** using `ioh.logger.Analyzer`.
4. Outputs full convergence traces (`.json` and `.dat`) and `provenance.json` records directly into `results/evaluations/`.

In [ ]:
import sys
import json
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from shared.config import DATA_DIR, PROJECT_ROOT, RESULTS_DIR
from benchmarking import BenchmarkEvaluationService

# ── User Execution Controls & Selective Filters ──────────────────────────────
FORCE_REEVALUATE  = False   # Set True to bypass cache and re-evaluate all
FILTER_MODELS     = None    # e.g., ['7b'], ['14b'], or None for all discovered models
FILTER_PROBLEMS   = None    # e.g., [1, 8, 11, 15, 21] or None for all
FILTER_STRATEGIES = None    # e.g., ['baseline', 'guided', 'thinking', 'vectorization'] or None
FILTER_MODES      = None    # e.g., ['clean'], ['noisy'], or None for all
FILTER_DIMS       = None    # e.g., [2, 3, 5], or None for all
N_RUNS            = 10      # Number of independent benchmark runs per configuration

CHAMPIONS_PATH   = DATA_DIR / 'champions.json'
EVALUATIONS_DIR  = RESULTS_DIR / 'evaluations' / 'traces'

service          = BenchmarkEvaluationService(
    DATA_DIR / 'db.sqlite3',
    EVALUATIONS_DIR,
    PROJECT_ROOT,
    n_runs=N_RUNS,
)

print(f'Evaluations Directory : {EVALUATIONS_DIR}')
print(f'Champions JSON Source : {CHAMPIONS_PATH}')

## 1. Load Champions JSON

In [ ]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f'Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 03 first.')

with open(CHAMPIONS_PATH, 'r', encoding='utf-8') as f:
    champions_raw = json.load(f)

champions_flat = service.champions_repo.get_champions_flat(champions_raw)

print(f'Loaded {len(champions_flat)} champion configuration(s) across {len(champions_raw)} model category(ies):')
for model_key, model_dict in champions_raw.items():
    if isinstance(model_dict, dict) and 'code_path' not in model_dict:
        print(f'  • {model_key:<45}: {len(model_dict):3d} champions')


## 2. Pre-Flight Diagnostic Dashboard: Coverage & Workload Audit

Scans the filesystem (`results/evaluations/`) and validates checksums against `data/champions.json` to categorize every champion as:
- **`COMPLETED`**: Valid `provenance.json` with matching SHA-256 code hash and complete `.dat` traces ($N=10$ runs).
- **`PENDING`**: Not yet evaluated.
- **`NEEDS_RERUN`**: Folder exists but files are incomplete, corrupted, or code hash changed.
- **`MISSING_CODE`**: Generated `.py` file missing on disk.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

def render_html_dashboard(
    df_audit: pd.DataFrame,
    title: str = "Benchmark Evaluation Pre-Flight Audit",
    subtitle: str = "Real-time status of empirical evaluation runs",
    target_runs: int = 10,
    group_column: str = "model",
) -> str:
    """Render responsive HTML pre-flight dashboard for Jupyter display."""
    if df_audit.empty:
        return "<div>No audit data available.</div>"

    grp_col = (
        group_column
        if group_column in df_audit.columns
        else ("baseline" if "baseline" in df_audit.columns else df_audit.columns[0])
    )

    summary_rows = []
    for name, grp in df_audit.groupby(grp_col):
        total = len(grp)
        completed = len(grp[grp["status"] == "COMPLETED"])
        pending = len(grp[grp["status"] == "PENDING"])
        needs_rerun = len(grp[grp["status"] == "NEEDS_RERUN"])
        missing_code = len(grp[grp["status"] == "MISSING_CODE"])
        to_run_mask = grp["status"].isin(["PENDING", "NEEDS_RERUN"])
        if "is_filtered" in grp.columns:
            to_run_mask = to_run_mask & (~grp["is_filtered"])
        to_run = len(grp[to_run_mask])
        pct = (completed / total * 100) if total > 0 else 0.0
        summary_rows.append({
            "Group": str(name).upper(),
            "Total Tasks": total,
            "Completed": completed,
            "Pending": pending,
            "Needs Rerun": needs_rerun,
            "Missing Code": missing_code,
            "Queue to Run": to_run,
            "Progress (%)": pct,
        })

    df_summary = pd.DataFrame(summary_rows)
    total_queue = int(df_summary["Queue to Run"].sum())
    total_completed = int(df_summary["Completed"].sum())
    total_tasks = int(df_summary["Total Tasks"].sum())
    overall_pct = (total_completed / total_tasks * 100) if total_tasks > 0 else 0.0

    html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                🚀 {title}
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">{subtitle}</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            Target: N={target_runs} runs / condition
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Targets</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{total_tasks}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Across {len(df_summary)} Categories</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{total_completed}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{overall_pct:.1f}% Overall Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{total_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">{total_queue * target_runs} Total Runs</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Est. Runtime</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">~{total_queue * 1.2:.0f}m</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">@ ~1.2s per run</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Completion Progress Breakdown
        </div>
"""
    for _, row in df_summary.iterrows():
        grp_name = row["Group"]
        tot = int(row["Total Tasks"])
        comp = int(row["Completed"])
        pend = int(row["Pending"])
        rerun = int(row["Needs Rerun"])
        q = int(row["Queue to Run"])
        pct = float(row["Progress (%)"])
        color = "#10B981" if pct > 75 else ("#3B82F6" if pct > 25 else "#F59E0B")
        rerun_pct = (rerun / tot * 100) if tot > 0 else 0

        html += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>⚙️ <strong style="color: #0F172A;">{grp_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
"""
    html += """
    </div>
</div>
"""
    return html

# Audit champions completion and validity status via BenchmarkEvaluationService
df_audit = service.audit_champions_workload(
    champions_flat,
    filter_models=FILTER_MODELS,
    filter_problems=FILTER_PROBLEMS,
    filter_strategies=FILTER_STRATEGIES,
    filter_modes=FILTER_MODES,
    filter_dims=FILTER_DIMS,
)

# Render styled pre-flight visual dashboard
html_dashboard = render_html_dashboard(
    df_audit,
    title="Champion Evaluation Pre-Flight Audit",
    target_runs=N_RUNS,
    group_column="model",
)
display(HTML(html_dashboard))


## 3. Execute Benchmark Evaluations (N=10 Independent Runs)

Executes all pending or rerun configurations. Already completed runs with valid provenance are skipped automatically unless `FORCE_REEVALUATE=True`.

In [ ]:
# Run all pending evaluations via BenchmarkEvaluationService
results_df = service.run_champions(
    champions_flat=champions_flat,
    filter_models=FILTER_MODELS,
    filter_problems=FILTER_PROBLEMS,
    filter_strategies=FILTER_STRATEGIES,
    filter_modes=FILTER_MODES,
    filter_dims=FILTER_DIMS,
    force_rerun=FORCE_REEVALUATE,
)